# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K.-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [52]:
import pandas as pd
import numpy as np
import pickle

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_validate, train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, make_scorer
import shap
from sklearn.metrics import mean_squared_error, r2_score, explained_variance_score

In [53]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [54]:
# Target: area (burned area in hectares)
# Apply log(1 + area) transform since area is heavily right-skewed with many zeros
Y = np.log1p(fires_dt['area'])

# Features: everything except area
X = fires_dt.drop(columns=['area'])

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
X.head()

X shape: (517, 12)
Y shape: (517,)


,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0


In [55]:
# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Identify column types
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f"Numeric columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Numeric columns: ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
Categorical columns: ['month', 'day']
Train size: 413, Test size: 104


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [56]:
# Preproc 1: StandardScaler for numerics, OneHotEncoder for categoricals
preproc1 = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist'), cat_cols)
])

preproc1

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [57]:
# Preproc 2: StandardScaler + PowerTransformer (Yeo-Johnson) for numerics, OneHotEncoder for categoricals
num_pipe_transformed = Pipeline([
    ('scaler', StandardScaler()),
    ('power', PowerTransformer(method='yeo-johnson'))
])

preproc2 = ColumnTransformer(transformers=[
    ('num', num_pipe_transformed, num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='infrequent_if_exist'), cat_cols)
])

preproc2

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [58]:
# Pipeline A = preproc1 + baseline
pipe_a = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', Ridge())
])
pipe_a

,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [59]:
# Pipeline B = preproc2 + baseline
pipe_b = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', Ridge())
])
pipe_b

,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [60]:
# Pipeline C = preproc1 + advanced model
pipe_c = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', GradientBoostingRegressor())
])
pipe_c

,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [61]:
# Pipeline D = preproc2 + advanced model

pipe_d = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', GradientBoostingRegressor())
])
pipe_d    

,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [62]:
# Define scoring metric (using neg_mean_squared_error for RMSE, which is appropriate for regression)
scoring = 'neg_mean_squared_error'  # We'll use RMSE (lower is better, so neg is used)

# Pipeline A: preproc1 + Ridge
param_grid_a = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0]  # Ridge regularization parameter
}

grid_a = GridSearchCV(
    estimator=pipe_a,
    param_grid=param_grid_a,
    scoring=scoring,
    cv=5,
    n_jobs=-1,
    refit=True
)
grid_a.fit(X_train, Y_train)
print("Pipeline A - Best params:", grid_a.best_params_)
print("Pipeline A - Best CV score (neg MSE):", grid_a.best_score_)

Pipeline A - Best params: {'regressor__alpha': 100.0}
Pipeline A - Best CV score (neg MSE): -2.1221038020564458


/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site

In [63]:
# Pipeline B: preproc2 + Ridge
param_grid_b = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0]
}

grid_b = GridSearchCV(
    estimator=pipe_b,
    param_grid=param_grid_b,
    scoring=scoring,
    cv=5,
    n_jobs=-1,
    refit=True
)
grid_b.fit(X_train, Y_train)
print("Pipeline B - Best params:", grid_b.best_params_)
print("Pipeline B - Best CV score (neg MSE):", grid_b.best_score_)

Pipeline B - Best params: {'regressor__alpha': 100.0}
Pipeline B - Best CV score (neg MSE): -1.917414326676289


/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site

In [64]:
# Pipeline C: preproc1 + GradientBoostingRegressor
param_grid_c = {
    'regressor__n_estimators': [50, 100, 150],
    'regressor__learning_rate': [0.01, 0.1, 0.2],
    'regressor__max_depth': [3, 5]
}
# This gives us 3 * 3 * 2 = 18 combinations (more than 4 required)

grid_c = GridSearchCV(
    estimator=pipe_c,
    param_grid=param_grid_c,
    scoring=scoring,
    cv=5,
    n_jobs=-1,
    refit=True
)
grid_c.fit(X_train, Y_train)
print("Pipeline C - Best params:", grid_c.best_params_)
print("Pipeline C - Best CV score (neg MSE):", grid_c.best_score_)

/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site

Pipeline C - Best params: {'regressor__learning_rate': 0.01, 'regressor__max_depth': 3, 'regressor__n_estimators': 50}
Pipeline C - Best CV score (neg MSE): -1.920404106596266


/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [65]:
# Pipeline D: preproc2 + GradientBoostingRegressor
param_grid_d = {
    'regressor__n_estimators': [50, 100, 150],
    'regressor__learning_rate': [0.01, 0.1, 0.2],
    'regressor__max_depth': [3, 5]
}

grid_d = GridSearchCV(
    estimator=pipe_d,
    param_grid=param_grid_d,
    scoring=scoring,
    cv=5,
    n_jobs=-1,
    refit=True
)
grid_d.fit(X_train, Y_train)
print("Pipeline D - Best params:", grid_d.best_params_)
print("Pipeline D - Best CV score (neg MSE):", grid_d.best_score_)

/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site

Pipeline D - Best params: {'regressor__learning_rate': 0.01, 'regressor__max_depth': 3, 'regressor__n_estimators': 50}
Pipeline D - Best CV score (neg MSE): -1.9201766457651896


/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/davidancor/Desktop/coding/DSI/production/assignment-2/production/production-env/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


# Evaluate

+ Which model has the best performance?

In [66]:
# Evaluate all models on test set
results = {}

for name, grid in [('A', grid_a), ('B', grid_b), ('C', grid_c), ('D', grid_d)]:
    y_pred = grid.predict(X_test)
    mse = mean_squared_error(Y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(Y_test, y_pred)
    r2 = r2_score(Y_test, y_pred)
    explained_var = explained_variance_score(Y_test, y_pred)
    
    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'Explained Variance': explained_var,
        'Best CV Score': grid.best_score_
    }

# Create comparison dataframe
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('RMSE') 
print(results_df)

       RMSE       MAE        R²  Explained Variance  Best CV Score
C  1.467065  1.188901  0.020739            0.021090      -1.920404
D  1.467069  1.188920  0.020733            0.021085      -1.920177
B  1.468326  1.191414  0.019055            0.021496      -1.917414
A  1.473051  1.192533  0.012732            0.015086      -2.122104


In [67]:
# Identify best model
best_model_name = results_df.index[0]
best_model_grid = {'A': grid_a, 'B': grid_b, 'C': grid_c, 'D': grid_d}[best_model_name]
best_model = best_model_grid.best_estimator_

print(f"\nBest performing model: Pipeline {best_model_name}")
print(f"Best RMSE: {results_df.loc[best_model_name, 'RMSE']:.4f}")
print(f"Best R²: {results_df.loc[best_model_name, 'R²']:.4f}")


Best performing model: Pipeline C
Best RMSE: 1.4671
Best R²: 0.0207


# Export

+ Save the best performing model to a pickle file.

In [68]:
# Save the best performing model
import os
os.makedirs('./models', exist_ok=True)

with open('./models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model_name, f)
    
print(f"Best model (Pipeline {best_model_name}) saved to ./models/best_model.pkl")

Best model (Pipeline C) saved to ./models/best_model.pkl


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [69]:
import shap

# Use Pipeline C (the best model)
best_model_grid = {'A': grid_a, 'B': grid_b, 'C': grid_c, 'D': grid_d}[best_model_name]
best_pipeline = best_model_grid.best_estimator_

# Transform the test data
X_test_transformed = best_pipeline.named_steps['preprocessing'].transform(X_test)
feature_names = best_pipeline.named_steps['preprocessing'].get_feature_names_out()

# Create SHAP explainer for GradientBoostingRegressor
explainer = shap.TreeExplainer(best_pipeline.named_steps['regressor'])

# Calculate SHAP values for first 100 test samples
shap_values = explainer.shap_values(X_test_transformed[:100])

In [70]:
obs_idx = 1  # 1 observation

print(f"Observation {obs_idx}:")
print(f"  Actual value: {Y_test.iloc[obs_idx]:.4f}")
print(f"  Predicted value: {best_pipeline.predict(X_test.iloc[[obs_idx]])[0]:.4f}")
print(f"\nFeature contributions (SHAP values) for this observation:")

Observation 1:
  Actual value: 0.0000
  Predicted value: 1.0839

Feature contributions (SHAP values) for this observation:


In [71]:
# Show which features matter most for this one observation
shap_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP Value': shap_values[obs_idx]
}).sort_values('SHAP Value', key=abs, ascending=False)

print(shap_df.head(10))

           Feature  SHAP Value
5         num__isi   -0.027144
0     num__coord_x    0.022869
3         num__dmc    0.019296
6        num__temp   -0.014523
20  cat__month_sep   -0.011628
11  cat__month_dec   -0.005808
25    cat__day_tue    0.003983
7          num__rh   -0.003547
1     num__coord_y   -0.001547
4          num__dc   -0.000647


In [72]:
# most and least important across all test samples
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': np.abs(shap_values).mean(axis=0)
}).sort_values('Importance', ascending=False)

print("Most important features (overall):")
print(feature_importance.head(10))
print("\nLeast important features (overall):")
print(feature_importance.tail(10))

Most important features (overall):
           Feature  Importance
3         num__dmc    0.051393
0     num__coord_x    0.034816
11  cat__month_dec    0.014725
6        num__temp    0.013537
5         num__isi    0.013334
20  cat__month_sep    0.013060
7          num__rh    0.005758
8        num__wind    0.002705
17  cat__month_may    0.002573
25    cat__day_tue    0.002291

Least important features (overall):
           Feature  Importance
15  cat__month_jun         0.0
18  cat__month_nov         0.0
19  cat__month_oct         0.0
14  cat__month_jul         0.0
12  cat__month_feb         0.0
22    cat__day_sat         0.0
10  cat__month_aug         0.0
24    cat__day_thu         0.0
9        num__rain         0.0
13  cat__month_jan         0.0


## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.